# T5 - Integration Testing and Demo Runbook

**Guardian Eye task:** T5

Run this notebook top-to-bottom on the target virtual machine. Configuration is centralized near the top, and heavy model work only occurs in explicitly marked execution cells.


## 1. Demo configuration

List 3-5 representative clips on the target VM. Include both violence and non-violence cases plus at least one difficult/low-quality clip.


In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
from typing import Any
import json
import time
import httpx

BASE_URL = "http://127.0.0.1:8000"
DEMO_CLIPS = [
    Path("data/incoming/violence_clear.mp4"),
    Path("data/incoming/non_violence_clear.mp4"),
    Path("data/incoming/crowded_ambiguous.mp4"),
]

for clip in DEMO_CLIPS:
    print(f"{clip}: exists={clip.exists()}")


## 2. API smoke-test helpers

Calls the local API, validates the integrated response contract, and never contacts a cloud service.


In [ ]:
REQUIRED_ANALYZE_FIELDS = {
    "verdict", "confidence", "threshold", "evidence_packet", "explanation",
    "guardrail_status", "incident_id", "recap_frame_paths", "limitations_note",
}

def analyze_via_api(clip: Path) -> dict[str, Any]:
    with clip.open("rb") as stream:
        response = httpx.post(
            f"{BASE_URL}/analyze",
            files={"file": (clip.name, stream, "video/mp4")},
            timeout=900,
        )
    response.raise_for_status()
    payload = response.json()
    missing = REQUIRED_ANALYZE_FIELDS - payload.keys()
    assert not missing, f"Analyze response missing: {sorted(missing)}"
    assert payload["verdict"] in {"violence", "non-violence"}
    assert 0 <= payload["confidence"] <= 1
    assert payload["incident_id"]
    assert 2 <= len(payload["explanation"].split(".")) <= 6
    return payload

def get_incident(incident_id: str) -> dict[str, Any]:
    response = httpx.get(f"{BASE_URL}/incidents/{incident_id}", timeout=30)
    response.raise_for_status()
    return response.json()

def search_incidents(query: str, top_k: int = 5) -> list[dict[str, Any]]:
    response = httpx.post(
        f"{BASE_URL}/incidents/search",
        json={"query": query, "top_k": top_k},
        timeout=60,
    )
    response.raise_for_status()
    return response.json()


## 3. End-to-end demo test

Analyzes every configured clip, verifies incident persistence, and checks that historical search returns records.


In [ ]:
def run_demo_suite(clips: list[Path]) -> list[dict[str, Any]]:
    missing = [str(clip) for clip in clips if not clip.is_file()]
    if missing:
        raise FileNotFoundError(f"Add demo clips before running: {missing}")

    results = []
    for clip in clips:
        started = time.perf_counter()
        analyzed = analyze_via_api(clip)
        stored = get_incident(analyzed["incident_id"])
        assert stored["incident_id"] == analyzed["incident_id"]
        assert stored["verdict"] == analyzed["verdict"]
        results.append({
            "clip": str(clip),
            "elapsed_seconds": round(time.perf_counter() - started, 2),
            **analyzed,
        })

    historical = search_incidents("show recent violence incidents", top_k=5)
    assert isinstance(historical, list)
    print(f"Historical query returned {len(historical)} records")
    return results

# Explicit target-VM execution:
# demo_results = run_demo_suite(DEMO_CLIPS)


## 4. Guardrail and response acceptance checks

Verifies the non-negotiable integration rules against collected demo results.


In [ ]:
def check_acceptance(result: dict[str, Any]) -> list[str]:
    failures = []
    if result["guardrail_status"] != "passed":
        failures.append("Narrator guardrail did not pass")
    if not result["evidence_packet"].get("packet_summary"):
        failures.append("Evidence packet summary missing")
    if not result["recap_frame_paths"]:
        failures.append("Visual recap paths missing")
    if "geometry" not in result["limitations_note"].lower():
        failures.append("Geometry-derived limitation is not disclosed")
    return failures

def acceptance_report(results: list[dict[str, Any]]) -> dict[str, Any]:
    rows = []
    for result in results:
        failures = check_acceptance(result)
        rows.append({"clip": result["clip"], "passed": not failures, "failures": failures})
    return {"passed": all(row["passed"] for row in rows), "clips": rows}

# report = acceptance_report(demo_results)
# print(json.dumps(report, indent=2))


## 5. Demo runbook

Use this sequence during the final team demo.

1. Confirm all model checkpoints and indexes are local; disable network access.
2. Start the API and call `/health`.
3. Select a representative clip and call `/analyze`.
4. Show classifier verdict/confidence first and state that they are final.
5. Show the deterministic evidence packet and geometry-derived caveat.
6. Show the grounded 2-4 sentence explanation and recap frames.
7. Open `/incidents/{id}` to prove the analyzed clip was persisted.
8. Ask a natural-language historical question through `/incidents/search`.
9. Show sequential runtime logs proving classifier unload occurred before VLM load.
10. Run the remaining clips and display the acceptance report.


## 6. Known limitations checklist

- The classifier is clip-level; exact timing and person attribution are geometry-derived estimates.
- The VLM narrates evidence but cannot change classifier verdict or confidence.
- Object/weapon claims require visible or telemetry-supported evidence.
- Performance depends on local GPU VRAM and disk speed.
- The 12 GB GPU flow requires strict sequential model loading.
- Reference and incident retrieval quality depends on the curated corpus and stored history.
- No cloud fallback is permitted during the demo.


## 7. Export demo results

After running the explicit demo cells on the target VM, export a compact JSON artifact for the team.


In [ ]:
def export_demo_report(results: list[dict[str, Any]], output_path="docs/demo_results.json") -> Path:
    output = Path(output_path)
    output.parent.mkdir(parents=True, exist_ok=True)
    report = {
        "acceptance": acceptance_report(results),
        "results": results,
    }
    output.write_text(json.dumps(report, indent=2), encoding="utf-8")
    return output

# export_demo_report(demo_results)
